<a href="https://colab.research.google.com/github/MGentieu/dl_project/blob/martin_nlp/starters/cv-project-starter/cv-project/notebooks/CV_tiny-imagenet-200.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Connexion à github et setup initial

Pour github : à exécuter dans le terminal de colab

```bash
git clone https://github.com/MGentieu/dl_project.git

### Step 0 — On confirme l'utilisation du GPU



In [1]:
!nvidia-smi || echo "nvidia-smi unavailable (CPU runtime)"

Fri Dec  5 16:20:50 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Step 1 : Installation et importation des bibliothèques

In [2]:
# Installation des librairies nécessaires
!pip -q install torch torchvision torchmetrics matplotlib tqdm ultralytics scikit-learn pyyaml

import os
import sys
import time
import copy
import zipfile
import urllib.request
import shutil
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import Counter
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score

# Configuration du Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilisation du device : {device}")

# Seeds pour la reproductibilité
torch.manual_seed(42)
np.random.seed(42)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 44.3 MB/s eta 0:00:00
Utilisation du device : cuda


### Step 2 — Pointer le répertoire du projet


In [3]:
# The user has provided the explicit path to the project root.
PROJECT_ROOT = Path("/content/dl_project/starters/cv-project-starter/cv-project")

print(f"Environment: Colab/Kaggle (remote server), using provided PROJECT_ROOT")

# Validate structure
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError(f"Missing src/ directory at {PROJECT_ROOT}")

# Setup Python path
os.chdir(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {Path.cwd()}")


Environment: Colab/Kaggle (remote server), using provided PROJECT_ROOT
Project root: /content/dl_project/starters/cv-project-starter/cv-project
Working directory: /content/dl_project/starters/cv-project-starter/cv-project


M1: Problem Scoping & Data Validation
1.1 Data Card — Tiny-ImageNet-200
Source : Sous-ensemble du célèbre dataset ImageNet (Stanford).

Contenu : 200 classes d'objets.

Volume :

Train : 100,000 images (500 par classe).

Validation : 10,000 images (50 par classe).

Test : 10,000 images (non labellisées, nous ne les utiliserons pas ici).

Résolution : Images couleur de 64x64 pixels.

Biais potentiels : Comme ImageNet, le dataset peut contenir des biais liés à la sélection des catégories (objets occidentaux, animaux spécifiques) et à la méthode de collecte web.

Input : Image RGB (3 canaux), redimensionnée à 224x224 pour ResNet.

Output : Probabilités pour 200 classes.

Métrique principale : Exactitude (Accuracy) sur le jeu de validation.

1.2 Téléchargement et Préparation des Données
Le code ci-dessous télécharge le dataset et restructure le dossier de validation pour qu'il soit compatible avec torchvision.datasets.ImageFolder (qui attend une structure dossier/classe/image.jpg).

In [4]:
# Téléchargement et préparation (Script fourni dans README_CV_ADVANCED.md)
url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
path = "data/tiny-imagenet-200.zip"
data_dir = "data"

if not os.path.exists(os.path.join(data_dir, "tiny-imagenet-200")):
    print("Téléchargement du dataset...")
    os.makedirs(data_dir, exist_ok=True)
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)

    print("Extraction...")
    with zipfile.ZipFile(path, "r") as z:
        z.extractall(data_dir)

    # Restructuration du dossier de validation
    print("Restructuration du dossier de validation...")
    val_dir = pathlib.Path(data_dir) / "tiny-imagenet-200/val"
    val_img_dir = val_dir / "images"

    # Lecture des annotations pour mapper image -> classe
    val_map = {}
    with open(val_dir / "val_annotations.txt", "r") as f:
        for line in f:
            parts = line.strip().split("\t")
            val_map[parts[0]] = parts[1] # image_file -> class_id

    # Déplacement des images dans des sous-dossiers par classe
    for img, wnid in val_map.items():
        target_dir = val_dir / wnid
        target_dir.mkdir(exist_ok=True)
        src = val_img_dir / img
        dst = target_dir / img
        if src.exists():
            shutil.move(src, dst)

    # Suppression du dossier images vide et du fichier txt inutile après déplacement
    if val_img_dir.exists() and not os.listdir(val_img_dir):
        os.rmdir(val_img_dir)

    print("Dataset prêt !")
else:
    print("Le dataset semble déjà présent.")

Téléchargement du dataset...
Extraction...
Restructuration du dossier de validation...
Dataset prêt !


# M2 : Baseline Model implementation

#### Code basique pour vérifier le bon fonctionnement et le chargement de Caltech-101

In [5]:
# Configuration inspirée de cv_tinyimagenet.yaml
IMG_SIZE = 224
BATCH_SIZE = 128
NUM_WORKERS = 2

# Transformations
# Train : Augmentation de données (Flip, Rotation, ColorJitter) + Resize
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), # Upscaling de 64x64 à 224x224
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Val : Juste Resize et Normalize
val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Création des Datasets
train_dir = os.path.join(data_dir, "tiny-imagenet-200/train")
val_dir = os.path.join(data_dir, "tiny-imagenet-200/val")

train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(root=val_dir, transform=val_transforms)

# Dataloaders
dataloaders = {
    'train': DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS),
    'val': DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
}

dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset)}
class_names = train_dataset.classes
num_classes = len(class_names)

print(f"Classes: {num_classes}")
print(f"Images Train: {dataset_sizes['train']}, Images Val: {dataset_sizes['val']}")

Classes: 200
Images Train: 100000, Images Val: 10000


In [6]:
def get_model(num_classes, freeze_backbone=True):
    # Chargement de ResNet18 pré-entraîné
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # 1. Freeze (Geler) le backbone si demandé
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # 2. Modification de la tête (Fully Connected layer)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)

    return model

baseline_model = get_model(num_classes, freeze_backbone=True)
baseline_model = baseline_model.to(device)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 69.4MB/s]


Le chargement du dataset et la transformation basique s'est bien effectuée. On passe donc à l'étape suivante, le chargement réel du dataset et l'exploration des données

On charge dans un premier temps le fichier yaml de configuration

In [7]:
# Récupération d'un batch
inputs, classes = next(iter(dataloaders['train']))
inputs = inputs.to(device)

# Passage dans le modèle
output = baseline_model(inputs)

print(f"Input shape: {inputs.shape}")   # Devrait être [128, 3, 224, 224]
print(f"Output shape: {output.shape}") # Devrait être [128, 200]
print("Test Batch: SUCCESS")

Input shape: torch.Size([128, 3, 224, 224])
Output shape: torch.Size([128, 200])
Test Batch: SUCCESS


In [9]:
criterion = nn.CrossEntropyLoss()
optimizer_base = optim.Adam(baseline_model.fc.parameters(), lr=1e-3)

print("Lancement de la baseline (1 époque)...")
# Boucle simplifiée pour M2
baseline_model.train()
running_loss = 0.0
for i, (inputs, labels) in enumerate(dataloaders['train']):
    inputs, labels = inputs.to(device), labels.to(device)

    optimizer_base.zero_grad()
    outputs = baseline_model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer_base.step()

    running_loss += loss.item()
    if i % 100 == 0:
        print(f"Batch {i}, Loss: {loss.item():.4f}")

print("Baseline training finished.")

Lancement de la baseline (1 époque)...
Batch 0, Loss: 3.4987


KeyboardInterrupt: 

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = np.inf
        self.early_stop = False

    def __call__(self, val_loss, model, path='best_model.pt'):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), path)
            print(f"Validation loss improved to {val_loss:.4f}. Model saved.")
        else:
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

def train_model(model, criterion, optimizer, scheduler, num_epochs=25, patience=5):
    since = time.time()
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    early_stopping = EarlyStopping(patience=patience)

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # Chaque époque a une phase d'entraînement et une phase de validation
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                # Forward
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward + Optimize uniquement si train
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            # Enregistrement historique
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.cpu().item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.cpu().item())

                # Check Early Stopping
                early_stopping(epoch_loss, model)

        if early_stopping.early_stop:
            print("Early stopping triggered")
            break

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')

    # Charger le meilleur modèle
    model.load_state_dict(torch.load('best_model.pt'))
    return model, history

1. Justification des choix techniques
Voici pourquoi ces paramètres ont été choisis pour Tiny-ImageNet-200 :

Le Modèle : ResNet18 (Pre-trained)

Pourquoi ? Tiny-ImageNet est plus complexe que CIFAR-10 mais plus petit qu'ImageNet complet. ResNet18 offre un excellent compromis : il est assez profond pour capter des features complexes sans être trop lourd pour être entraîné sur une instance Colab (contrairement à ResNet50 ou 101 qui seraient plus lents).

Transfer Learning : Utiliser les poids IMAGENET1K_V1 est crucial. Le modèle a déjà "vu" des millions d'images similaires. Geler le "backbone" au début permet d'entraîner uniquement la dernière couche de classification sans détruire les filtres déjà appris.

L'Optimiseur : AdamW

Pourquoi ? C'est l'standard actuel pour le Deep Learning moderne (vision et NLP). Contrairement à SGD (qui demande souvent beaucoup de réglages manuels du momentum), AdamW gère mieux le taux d'apprentissage adaptatif par paramètre et découple proprement la régularisation (Weight Decay), ce qui aide à la généralisation.

Le Learning Rate (Taux d'apprentissage) : 5e-4 (0.0005)

Pourquoi ? C'est une valeur "sûre" pour AdamW lors d'un Transfer Learning.

Si on entraîne tout le réseau (Fine-tuning), on baisserait souvent à 1e-5.

Si c'était trop haut (1e-2), le modèle divergerait. Si trop bas (1e-6), l'entraînement stagnerait.

Nombre d'époques : 30 (avec Early Stopping)

Pourquoi ? 30 époques suffisent généralement pour qu'un ResNet converge sur ce type de dataset. L'Early Stopping (patience=7 dans ton YAML) est là pour arrêter l'entraînement si la val_loss remonte, évitant ainsi de perdre du temps et de l'énergie (et l'overfitting).

In [ ]:
print("\n=== Expérience 1 : Frozen Backbone ===")
model_exp1 = get_model(num_classes, freeze_backbone=True)
model_exp1 = model_exp1.to(device)

# Optimiseur avec Weight Decay (L2 Regularization)
optimizer1 = optim.AdamW(model_exp1.fc.parameters(), lr=5e-4, weight_decay=1e-2)
# Scheduler Cosine
scheduler1 = optim.lr_scheduler.CosineAnnealingLR(optimizer1, T_max=10)

model_exp1, hist1 = train_model(model_exp1, criterion, optimizer1, scheduler1, num_epochs=10, patience=3)

In [ ]:
print("\n=== Expérience 2 : Full Fine-Tuning ===")
# On dégèle toutes les couches
for param in model_exp1.parameters():
    param.requires_grad = True

# Learning rate plus bas pour ne pas casser les features apprises
optimizer2 = optim.AdamW(model_exp1.parameters(), lr=1e-5, weight_decay=1e-2)
scheduler2 = optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=10)

model_final, hist2 = train_model(model_exp1, criterion, optimizer2, scheduler2, num_epochs=10, patience=3)

In [ ]:
results_df = pd.DataFrame({
    'Expérience': ['Frozen Head Only', 'Fine-Tuning'],
    'Best Val Acc': [max(hist1['val_acc']), max(hist2['val_acc'])],
    'Final Val Loss': [hist1['val_loss'][-1], hist2['val_loss'][-1]]
})
display(results_df)

In [ ]:
def evaluate_model_detailed(model, dataloader):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    return np.array(y_true), np.array(y_pred)

# Evaluation
y_true, y_pred = evaluate_model_detailed(model_final, dataloaders['val'])

# Rapport de classification
print(classification_report(y_true, y_pred, digits=4))

# Matrice de confusion (Sur un sous-ensemble de classes pour lisibilité)
# On affiche les 20 premières classes
plt.figure(figsize=(12, 10))
cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm[:20, :20], annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix (First 20 classes)")
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()